# 01. EDA — Batch 1 (학습 데이터)

- 대상: Batch 1 (2017-05-12), 셀 46개
- 사전 준비: 터미널에서 `.venv/bin/python src/preprocess.py 1` 한 번 실행 → `data/cache/`에 캐시 생성
- 각 섹션: **그래프 → 관찰 → 시사점(모델링 전략)** 순서로 채워 넣기

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
plt.rcParams.update({'figure.figsize': (12, 5), 'font.size': 12, 'font.family': 'AppleGothic', 'axes.unicode_minus': False})

df = pd.read_csv('../data/cache/batch1_summary.csv')
z = np.load('../data/cache/batch1_dq.npz')
V = np.linspace(3.5, 2.0, 1000)           # Vdlin (전압 축)
cells = df.drop_duplicates('cell')[['cell', 'cycle_life', 'policy']].reset_index(drop=True)
print(df.shape, '| 셀 수:', len(cells), '| 정책 수:', cells.policy.nunique())
df.head()

## Q1. Cycle Life 분포

In [ ]:
cl = cells.cycle_life
fig, ax = plt.subplots(1, 2)
ax[0].hist(cl, bins=15, color='steelblue', edgecolor='white')
ax[0].axvline(cl.mean(), color='tomato', ls='--', label=f'mean={cl.mean():.0f}'); ax[0].legend()
ax[0].set_xlabel('cycle_life'); ax[0].set_ylabel('셀 수')
ax[1].boxplot(cl, orientation='horizontal'); ax[1].set_xlabel('cycle_life')
plt.show()

print(cl.describe().round(0).to_string())
print(f"\n장수명(>1000): {(cl>1000).sum()}개 ({(cl>1000).mean():.0%})")
print(f"단수명(<500) : {(cl<500).sum()}개")
print(f"550 미만     : {(cl<550).sum()}개  (논문 분류 기준 550)")
print(f"IQR 기준 이상치:", cells[(cl < cl.quantile(.25)-1.5*(cl.quantile(.75)-cl.quantile(.25))) | (cl > cl.quantile(.75)+1.5*(cl.quantile(.75)-cl.quantile(.25)))].to_string())

**관찰**
- (작성)

**시사점**
- (작성: 분류 기준 550을 쓰면 클래스 비율이 어떻게 되는지 → 회귀 vs 분류 판단 근거)

## Q2. 열화 곡선 (방전 용량 QD)

In [ ]:
nom = df[df.cycle <= 5].groupby('cell').QDischarge.median().median()
clean = df[df.QDischarge.between(nom*0.8, nom*1.2)]      # 초기 스파이크 제거
print(f'공칭 용량 {nom:.3f} Ah | 제거 행 {len(df)-len(clean)} ({(len(df)-len(clean))/len(df):.2%})')

fig, ax = plt.subplots(figsize=(13, 6))
cmap = plt.cm.coolwarm_r; lo, hi = cells.cycle_life.min(), cells.cycle_life.max()
for c, g in clean.groupby('cell'):
    ax.plot(g.cycle, g.QDischarge, color=cmap((g.cycle_life.iloc[0]-lo)/(hi-lo)), lw=.8, alpha=.7)
ax.axhline(nom*0.8, color='red', ls='--', label='EOL (80%)')
ax.set_ylim(0.7, 1.2); ax.set_xlabel('cycle'); ax.set_ylabel('QD (Ah)'); ax.legend(); plt.show()

In [ ]:
# 열화 가속 여부: 수명 대비 구간별 용량 감소율, knee 후보 = 기울기가 처음으로 2배 이상 가팔라지는 지점
def knee(g, w=50):
    q = g.QDischarge.rolling(w, center=True, min_periods=w//2).mean().values
    s = -np.gradient(q)                                   # 사이클당 용량 감소
    base = np.median(s[w:200]) if len(s) > 200 else np.nan
    idx = np.where((np.arange(len(s)) > 200) & (s > 2*base))[0]
    return idx[0]+1 if len(idx) else np.nan

kn = clean.groupby('cell').apply(knee).rename('knee')
k = cells.merge(kn, left_on='cell', right_index=True)
k['knee_ratio'] = k.knee / k.cycle_life
print(f"knee 탐지 셀: {k.knee.notna().sum()}/{len(k)}  | knee/수명 중앙값: {k.knee_ratio.median():.2f}")
plt.scatter(k.knee, k.cycle_life); plt.xlabel('knee 후보 사이클'); plt.ylabel('cycle_life'); plt.show()

**관찰**
- (작성: 초기 용량이 균일한가? 열화가 선형인가 가속되는가? knee가 있는가?)

**시사점**
- (작성: 초기 QD만으로는 구분이 안 보임 → ΔQ(V) 필요 등)

## Q3. ΔQ(V) 곡선 (사이클 100 − 사이클 10)

In [ ]:
dq = z['q100'] - z['q10']                      # (46, 1000)
life = cells.cycle_life.values
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
for i in range(len(dq)):
    ax[0].plot(V, dq[i], color=cmap((life[i]-lo)/(hi-lo)), lw=.9, alpha=.8)
ax[0].set_xlabel('V'); ax[0].set_ylabel('ΔQ100-10(V) [Ah]'); ax[0].set_title('색: 파랑=장수명, 빨강=단수명')

order = np.argsort(life)
for name, idx, col in [('단수명 5개', order[:5], 'tomato'), ('장수명 5개', order[-5:], 'steelblue')]:
    ax[1].plot(V, dq[idx].mean(0), color=col, lw=2.5, label=name)
ax[1].set_xlabel('V'); ax[1].legend(); plt.show()

In [ ]:
# 곡선 → 숫자 하나로: 논문의 핵심 feature = log10(|분산|), log10(|최솟값|)
feat = pd.DataFrame({'cell': cells.cell, 'cycle_life': life,
                     'dQ_log_var': np.log10(np.abs(np.nanvar(dq, axis=1))),
                     'dQ_log_min': np.log10(np.abs(np.nanmin(dq, axis=1)))})
fig, ax = plt.subplots(1, 2, figsize=(13, 5))
for a, c in zip(ax, ['dQ_log_var', 'dQ_log_min']):
    a.scatter(feat[c], feat.cycle_life, edgecolors='white'); a.set_xlabel(c); a.set_ylabel('cycle_life')
    a.set_title(f"Pearson r = {feat[c].corr(feat.cycle_life):.3f}  |  Spearman = {feat[c].corr(feat.cycle_life, method='spearman'):.3f}")
plt.show()

**관찰**
- (작성: 장/단수명 곡선 모양 차이, 두 통계값의 상관)

**시사점**
- (작성: feature 후보로 채택 여부)

## Q4. 충전 조건(C-rate)과 수명

In [ ]:
pl = cells.groupby('policy').cycle_life.agg(['mean', 'std', 'count']).sort_values('mean', ascending=False)
fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(range(len(pl)), pl['mean'], yerr=pl['std'].fillna(0), color='steelblue', alpha=.75, capsize=3)
ax.set_xticks(range(len(pl))); ax.set_xticklabels(pl.index, rotation=60, ha='right', fontsize=9)
ax.set_ylabel('평균 cycle_life'); plt.tight_layout(); plt.show()
pl

In [ ]:
# 정책 문자열 → 숫자: 1단계 C-rate, 전환 SOC(%), 2단계 C-rate, 평균 C-rate(≈ 충전속도)
import re
def parse(p):
    a, soc, b = re.match(r'([\d.]+)C\((\d+)%\)-([\d.]+)C', p).groups()
    a, soc, b = float(a), float(soc), float(b)
    return pd.Series({'c1': a, 'soc': soc, 'c2': b, 'c_avg': a*soc/100 + b*(1-soc/100)})
cells = cells.join(cells.policy.apply(parse))
print(cells[['c1', 'soc', 'c2', 'c_avg', 'cycle_life']].corr()['cycle_life'].round(3).to_string())
plt.scatter(cells.c_avg, cells.cycle_life); plt.xlabel('평균 C-rate(근사)'); plt.ylabel('cycle_life'); plt.show()

**관찰**
- (작성: 고속충전이 정말 짧은 수명인가? 같은 정책 내 편차는?)

**시사점**
- (작성: 정책 정보만으로 충분한가, 셀 신호가 추가로 필요한가)

## Q5. 초기 사이클 feature와 수명의 상관 (초기 100 사이클)

In [ ]:
e = df[df.cycle <= 100].groupby('cell')
X = pd.DataFrame({
    'QD_mean': e.QDischarge.mean(), 'QD_std': e.QDischarge.std(),
    'IR_mean': e.IR.mean(), 'Tavg_mean': e.Tavg.mean(), 'Tmax_mean': e.Tmax.mean(),
    'chargetime_mean': e.chargetime.mean()})
X['QD_slope'] = e.apply(lambda g: np.polyfit(g.cycle, g.QDischarge, 1)[0])   # 초기 용량 감소 기울기
X = X.join(feat.set_index('cell')[['dQ_log_var', 'dQ_log_min']]).join(cells.set_index('cell')[['c_avg']])
X['cycle_life'] = cells.set_index('cell').cycle_life
r = X.corr()
print(r.cycle_life.drop('cycle_life').sort_values(key=abs, ascending=False).round(3).to_string())

fig, ax = plt.subplots(figsize=(9, 7)); im = ax.imshow(r, cmap='RdBu_r', vmin=-1, vmax=1); plt.colorbar(im)
ax.set_xticks(range(len(r))); ax.set_xticklabels(r.columns, rotation=60, ha='right'); ax.set_yticks(range(len(r))); ax.set_yticklabels(r.columns)
for i in range(len(r)):
    for j in range(len(r)): ax.text(j, i, f'{r.iloc[i,j]:.1f}', ha='center', va='center', fontsize=8)
plt.show()

In [ ]:
# 다중공선성: |r| > 0.9 인 feature 쌍
f = r.drop(index='cycle_life', columns='cycle_life'); pairs = [(a, b, round(f.loc[a, b], 2)) for i, a in enumerate(f.index) for b in f.columns[i+1:] if abs(f.loc[a, b]) > .9]
pairs

**관찰**
- (작성: 수명과 가장 강한 feature, 서로 겹치는 feature)

**시사점**
- (작성: 어떤 feature를 쓰고 어떤 건 뺄지 → 선형+규제 모델 vs 트리 모델 근거)

---
# Batch 2 · 3 EDA (Batch 1과 비교)

- **목적: Batch 1과 얼마나 다른지 확인.** 모델 학습·feature 선택·튜닝에는 Batch 2/3를 쓰지 않음 (Batch 2는 Day 2 최종 평가 전용, Batch 3는 EDA만)
- 구성: 공통 준비 → **Batch 2 vs Batch 1** → **Batch 3 vs Batch 1**
- 사전 준비: `.venv/bin/python src/preprocess.py 2`, `... 3`

In [ ]:
# 공통 준비: 3개 배치 로드 + feature 계산 (이후 섹션에서 재사용)
from scipy.stats import ks_2samp
def load(n):
    d = pd.read_csv(f'../data/cache/batch{n}_summary.csv'); zz = np.load(f'../data/cache/batch{n}_dq.npz')
    c = d.drop_duplicates('cell')[['batch', 'cell', 'cycle_life', 'policy']].reset_index(drop=True)
    dq_ = zz['q100'] - zz['q10']
    c['dQ_log_var'] = np.log10(np.abs(np.nanvar(dq_, axis=1))); c['dQ_log_min'] = np.log10(np.abs(np.nanmin(dq_, axis=1)))
    c['n_cycles'] = d.groupby('cell').cycle.max().values
    return d, c, zz, dq_
B = {n: load(n) for n in (1, 2, 3)}
col = {1: 'steelblue', 2: 'tomato', 3: 'seagreen'}
cb_all = pd.concat([B[n][1] for n in B])
cb = cb_all.dropna(subset=['cycle_life']).copy()            # 수명 없는 셀(NaN)은 분석에서 제외

def parse_safe(p):
    try: return parse(p)
    except Exception: return pd.Series({'c1': np.nan, 'soc': np.nan, 'c2': np.nan, 'c_avg': np.nan})
cb = cb.reset_index(drop=True); cb = cb.join(cb.policy.apply(parse_safe))

def slope(g): g = g[(g.cycle >= 20) & (g.cycle <= 100)]; return np.polyfit(g.cycle, g.QDischarge, 1)[0]
rows = []
for n in B:
    d_ = B[n][0]; d_ = d_[d_.QDischarge.between(0.8, 1.2)]
    c_ = B[n][1].set_index('cell'); c_['slope'] = d_.groupby('cell').apply(slope); c_['knee'] = d_.groupby('cell').apply(knee)
    c_['knee_ratio'] = c_.knee / c_.cycle_life; rows.append(c_.dropna(subset=['cycle_life']))
sk = pd.concat(rows)

def early_feats(d, c):
    e = d[d.cycle <= 100].groupby('cell')
    X_ = pd.DataFrame({'QD_mean': e.QDischarge.mean(), 'IR_mean': e.IR.mean(), 'Tavg_mean': e.Tavg.mean(), 'chargetime_mean': e.chargetime.mean(),
                       'QD_slope': e.apply(lambda g: np.polyfit(g.cycle, g.QDischarge, 1)[0])})
    return X_.join(c.set_index('cell')[['dQ_log_var', 'dQ_log_min', 'cycle_life']]).dropna(subset=['cycle_life'])
corr_b = pd.DataFrame({f'Batch {n}': early_feats(B[n][0], B[n][1]).corr().cycle_life.drop('cycle_life') for n in B})
lo_, hi_ = cb.cycle_life.min(), cb.cycle_life.max()         # 색상 기준(전 배치 공통)

print('[데이터 품질] 배치별 요약')
display(cb_all.groupby('batch').agg(셀수=('cell', 'count'), 수명없음_NaN=('cycle_life', lambda s: s.isna().sum()), 최소사이클수=('n_cycles', 'min'), 정책수=('policy', 'nunique')))
print('수명이 NaN인 셀 = EOL(80%)에 도달하지 못했거나 특수 실험 셀 → 분석에서 제외')
display(cb_all[cb_all.cycle_life.isna()][['batch', 'cell', 'policy', 'n_cycles']])

---
# 🟥 Batch 2 vs Batch 1

> **Batch 2 = Day 2 최종 평가(Test) 데이터.** 아래는 분포 비교용이며, 결과를 보고 feature·모델을 고르지 않는다.

In [ ]:
T = 2                                                   # 비교 대상 배치
S = cb[cb.batch.isin([1, T])]
print(f'[핵심 숫자] Batch 1 vs Batch {T}')
display(S.groupby('batch').agg(셀수=('cell', 'count'), 평균수명=('cycle_life', 'mean'), 최소=('cycle_life', 'min'), 최대=('cycle_life', 'max'),
    장수명비율_1000초과=('cycle_life', lambda s: (s > 1000).mean()), 단수명비율_500미만=('cycle_life', lambda s: (s < 500).mean()),
    dQ와_수명_상관=('cell', lambda x: S.loc[x.index].dQ_log_var.corr(S.loc[x.index].cycle_life)),
    평균Crate와_수명_상관=('cell', lambda x: S.loc[x.index].c_avg.corr(S.loc[x.index].cycle_life))).round(2))

## [B2] Q1. Cycle Life 분포

In [ ]:
T = 2
fig, ax = plt.subplots(1, 2, figsize=(14, 5)); bins = np.linspace(300, 2000, 35)
for n in (1, T):
    v = cb[cb.batch == n].cycle_life; ax[0].hist(v, bins=bins, alpha=.55, color=col[n], label=f'Batch {n} (n={len(v)})')
ax[0].set_xlabel('cycle_life'); ax[0].set_ylabel('셀 수'); ax[0].legend()
ax[1].boxplot([cb[cb.batch == n].cycle_life for n in (1, T)], tick_labels=['Batch 1', f'Batch {T}']); ax[1].set_ylabel('cycle_life')
plt.show()
s = ks_2samp(cb[cb.batch == 1].cycle_life, cb[cb.batch == T].cycle_life); print(f'KS test Batch1 vs Batch{T}: stat={s.statistic:.2f}, p={s.pvalue:.1e}  (p가 작을수록 분포가 다름)')
print(f'분류 기준 550 → 단수명 비율: Batch 1 {(cb[cb.batch==1].cycle_life<550).mean():.0%} / Batch {T} {(cb[cb.batch==T].cycle_life<550).mean():.0%}')
print(f'\n[Batch {T} 수명 상위 5 셀]'); display(cb[cb.batch == T].nlargest(5, 'cycle_life')[['cell', 'cycle_life', 'policy']])
print(f'[Batch {T} 수명 하위 5 셀]'); display(cb[cb.batch == T].nsmallest(5, 'cycle_life')[['cell', 'cycle_life', 'policy']])

**관찰**
- (작성)

**시사점**
- (작성)

## [B2] Q2. 열화 곡선

In [ ]:
T = 2
fig, axs = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for a, n in zip(axs, (1, T)):
    d_ = B[n][0]; nom_ = d_[d_.cycle <= 5].groupby('cell').QDischarge.median().median()
    cl_ = d_[d_.QDischarge.between(nom_*0.8, nom_*1.2)]
    for c_, g_ in cl_.groupby('cell'):
        life_ = g_.cycle_life.iloc[0]
        a.plot(g_.cycle, g_.QDischarge, lw=.8, alpha=.7, color='lightgray' if np.isnan(life_) else cmap((life_-lo_)/(hi_-lo_)))
    a.axhline(nom_*0.8, color='red', ls='--'); a.set_title(f'Batch {n} (공칭 {nom_:.2f} Ah)'); a.set_xlabel('cycle'); a.set_ylim(0.7, 1.2)
axs[0].set_ylabel('QD (Ah)'); plt.show()
display(sk[sk.batch.isin([1, T])].groupby('batch').agg(초기기울기_중앙값=('slope', 'median'), knee_중앙값=('knee', 'median'), knee_수명비_중앙값=('knee_ratio', 'median')).round(4))

**관찰**
- (작성)

**시사점**
- (작성)

## [B2] Q3. ΔQ(V) 곡선

In [ ]:
T = 2
fig, axs = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for a, n in zip(axs, (1, T)):
    c_ = B[n][1]; dq_ = B[n][3]
    for i in range(len(dq_)):
        l_ = c_.cycle_life[i]; a.plot(V, dq_[i], lw=.9, alpha=.8, color='lightgray' if np.isnan(l_) else cmap((l_-lo_)/(hi_-lo_)))
    a.set_title(f'Batch {n}'); a.set_xlabel('V')
axs[0].set_ylabel('ΔQ100-10(V) [Ah]'); plt.show()

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
for n in (1, T):
    s = cb[cb.batch == n]; r_ = s.dQ_log_var.corr(s.cycle_life)
    ax[0].scatter(s.dQ_log_var, s.cycle_life, c=col[n], alpha=.75, edgecolors='white', label=f'Batch {n} (r={r_:.2f})')
    ax[1].plot(V, np.nanmedian(B[n][2]['q10'], axis=0), color=col[n], lw=2, label=f'Batch {n}')
ax[0].set_xlabel('dQ_log_var'); ax[0].set_ylabel('cycle_life'); ax[0].legend(); ax[0].set_title('feature-수명 관계가 배치 간 유지되는가')
ax[1].set_xlabel('V'); ax[1].set_ylabel('Q10(V) 중앙값 [Ah]'); ax[1].legend(); ax[1].set_title('Qdlin 단순 비교 시 왜곡 확인 (사이클 10)')
plt.show()

**관찰**
- (작성)

**시사점**
- (작성)

## [B2] Q4. 충전 조건(C-rate)

In [ ]:
T = 2
fig, ax = plt.subplots(figsize=(8, 5))
for n in (1, T):
    s = cb[cb.batch == n].dropna(subset=['c_avg']); ax.scatter(s.c_avg, s.cycle_life, c=col[n], alpha=.75, edgecolors='white', label=f'Batch {n} (r={s.c_avg.corr(s.cycle_life):.2f})')
ax.set_xlabel('평균 C-rate(근사)'); ax.set_ylabel('cycle_life'); ax.legend(); plt.show()
display(cb[cb.batch.isin([1, T])].groupby('batch').agg(정책수=('policy', 'nunique'), c_avg_평균=('c_avg', 'mean'), c_avg_최소=('c_avg', 'min'), c_avg_최대=('c_avg', 'max')).round(2))
print(f'Batch 1·{T} 공통 정책 수:', len(set(cb[cb.batch == 1].policy) & set(cb[cb.batch == T].policy)))
print(f'Batch {T} 정책별 평균 수명:'); display(cb[cb.batch == T].groupby('policy').cycle_life.agg(['mean', 'count']).sort_values('mean', ascending=False).round(0))

**관찰**
- (작성)

**시사점**
- (작성)

## [B2] Q5. 상관관계

In [ ]:
T = 2
display(corr_b[['Batch 1', f'Batch {T}']].assign(차이=lambda x: x[f'Batch {T}'] - x['Batch 1']).round(2).style.background_gradient(cmap='RdBu_r', vmin=-1, vmax=1, subset=['Batch 1', f'Batch {T}']))

**관찰**
- (작성: 부호·크기가 배치 간 일관된 feature vs 흔들리는 feature)

**시사점**
- (작성)

---
# 🟩 Batch 3 vs Batch 1

> **Batch 3 = 선택 과제(추가 평가), 이번에는 EDA만.** 과제 설명상 확인할 포인트: ① 수명 분포가 다름 ② 충전 곡선 시작점이 달라 `Qdlin` 단순 비교 시 왜곡 ③ 일부 셀 데이터 품질 문제(수명 NaN 셀 제외)

In [ ]:
T = 3                                                   # 비교 대상 배치
S = cb[cb.batch.isin([1, T])]
print(f'[핵심 숫자] Batch 1 vs Batch {T}')
display(S.groupby('batch').agg(셀수=('cell', 'count'), 평균수명=('cycle_life', 'mean'), 최소=('cycle_life', 'min'), 최대=('cycle_life', 'max'),
    장수명비율_1000초과=('cycle_life', lambda s: (s > 1000).mean()), 단수명비율_500미만=('cycle_life', lambda s: (s < 500).mean()),
    dQ와_수명_상관=('cell', lambda x: S.loc[x.index].dQ_log_var.corr(S.loc[x.index].cycle_life)),
    평균Crate와_수명_상관=('cell', lambda x: S.loc[x.index].c_avg.corr(S.loc[x.index].cycle_life))).round(2))

## [B3] Q1. Cycle Life 분포

In [ ]:
T = 3
fig, ax = plt.subplots(1, 2, figsize=(14, 5)); bins = np.linspace(300, 2000, 35)
for n in (1, T):
    v = cb[cb.batch == n].cycle_life; ax[0].hist(v, bins=bins, alpha=.55, color=col[n], label=f'Batch {n} (n={len(v)})')
ax[0].set_xlabel('cycle_life'); ax[0].set_ylabel('셀 수'); ax[0].legend()
ax[1].boxplot([cb[cb.batch == n].cycle_life for n in (1, T)], tick_labels=['Batch 1', f'Batch {T}']); ax[1].set_ylabel('cycle_life')
plt.show()
s = ks_2samp(cb[cb.batch == 1].cycle_life, cb[cb.batch == T].cycle_life); print(f'KS test Batch1 vs Batch{T}: stat={s.statistic:.2f}, p={s.pvalue:.1e}  (p가 작을수록 분포가 다름)')
print(f'분류 기준 550 → 단수명 비율: Batch 1 {(cb[cb.batch==1].cycle_life<550).mean():.0%} / Batch {T} {(cb[cb.batch==T].cycle_life<550).mean():.0%}')
print(f'\n[Batch {T} 수명 상위 5 셀]'); display(cb[cb.batch == T].nlargest(5, 'cycle_life')[['cell', 'cycle_life', 'policy']])
print(f'[Batch {T} 수명 하위 5 셀]'); display(cb[cb.batch == T].nsmallest(5, 'cycle_life')[['cell', 'cycle_life', 'policy']])

**관찰**
- (작성)

**시사점**
- (작성)

## [B3] Q2. 열화 곡선

In [ ]:
T = 3
fig, axs = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for a, n in zip(axs, (1, T)):
    d_ = B[n][0]; nom_ = d_[d_.cycle <= 5].groupby('cell').QDischarge.median().median()
    cl_ = d_[d_.QDischarge.between(nom_*0.8, nom_*1.2)]
    for c_, g_ in cl_.groupby('cell'):
        life_ = g_.cycle_life.iloc[0]
        a.plot(g_.cycle, g_.QDischarge, lw=.8, alpha=.7, color='lightgray' if np.isnan(life_) else cmap((life_-lo_)/(hi_-lo_)))
    a.axhline(nom_*0.8, color='red', ls='--'); a.set_title(f'Batch {n} (공칭 {nom_:.2f} Ah)'); a.set_xlabel('cycle'); a.set_ylim(0.7, 1.2)
axs[0].set_ylabel('QD (Ah)'); plt.show()
display(sk[sk.batch.isin([1, T])].groupby('batch').agg(초기기울기_중앙값=('slope', 'median'), knee_중앙값=('knee', 'median'), knee_수명비_중앙값=('knee_ratio', 'median')).round(4))

**관찰**
- (작성)

**시사점**
- (작성)

## [B3] Q3. ΔQ(V) 곡선

In [ ]:
T = 3
fig, axs = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for a, n in zip(axs, (1, T)):
    c_ = B[n][1]; dq_ = B[n][3]
    for i in range(len(dq_)):
        l_ = c_.cycle_life[i]; a.plot(V, dq_[i], lw=.9, alpha=.8, color='lightgray' if np.isnan(l_) else cmap((l_-lo_)/(hi_-lo_)))
    a.set_title(f'Batch {n}'); a.set_xlabel('V')
axs[0].set_ylabel('ΔQ100-10(V) [Ah]'); plt.show()

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
for n in (1, T):
    s = cb[cb.batch == n]; r_ = s.dQ_log_var.corr(s.cycle_life)
    ax[0].scatter(s.dQ_log_var, s.cycle_life, c=col[n], alpha=.75, edgecolors='white', label=f'Batch {n} (r={r_:.2f})')
    ax[1].plot(V, np.nanmedian(B[n][2]['q10'], axis=0), color=col[n], lw=2, label=f'Batch {n}')
ax[0].set_xlabel('dQ_log_var'); ax[0].set_ylabel('cycle_life'); ax[0].legend(); ax[0].set_title('feature-수명 관계가 배치 간 유지되는가')
ax[1].set_xlabel('V'); ax[1].set_ylabel('Q10(V) 중앙값 [Ah]'); ax[1].legend(); ax[1].set_title('Qdlin 단순 비교 시 왜곡 확인 (사이클 10)')
plt.show()

**관찰**
- (작성)

**시사점**
- (작성)

## [B3] Q4. 충전 조건(C-rate)

In [ ]:
T = 3
fig, ax = plt.subplots(figsize=(8, 5))
for n in (1, T):
    s = cb[cb.batch == n].dropna(subset=['c_avg']); ax.scatter(s.c_avg, s.cycle_life, c=col[n], alpha=.75, edgecolors='white', label=f'Batch {n} (r={s.c_avg.corr(s.cycle_life):.2f})')
ax.set_xlabel('평균 C-rate(근사)'); ax.set_ylabel('cycle_life'); ax.legend(); plt.show()
display(cb[cb.batch.isin([1, T])].groupby('batch').agg(정책수=('policy', 'nunique'), c_avg_평균=('c_avg', 'mean'), c_avg_최소=('c_avg', 'min'), c_avg_최대=('c_avg', 'max')).round(2))
print(f'Batch 1·{T} 공통 정책 수:', len(set(cb[cb.batch == 1].policy) & set(cb[cb.batch == T].policy)))
print(f'Batch {T} 정책별 평균 수명:'); display(cb[cb.batch == T].groupby('policy').cycle_life.agg(['mean', 'count']).sort_values('mean', ascending=False).round(0))

**관찰**
- (작성)

**시사점**
- (작성)

## [B3] Q5. 상관관계

In [ ]:
T = 3
display(corr_b[['Batch 1', f'Batch {T}']].assign(차이=lambda x: x[f'Batch {T}'] - x['Batch 1']).round(2).style.background_gradient(cmap='RdBu_r', vmin=-1, vmax=1, subset=['Batch 1', f'Batch {T}']))

**관찰**
- (작성: 부호·크기가 배치 간 일관된 feature vs 흔들리는 feature)

**시사점**
- (작성)

## 종합 → 모델 전략 메모
| EDA 발견 | 시사점 | 모델링 전략 |
|---|---|---|
| | | |

---
## 📌 팀원 공유 메모 (확인 후 삭제)

### 알아 둘 점
- **Batch 2는 최종 평가 전용.** feature 선택·하이퍼파라미터 튜닝·모델 선택은 Batch 1만 보고 한다. 위 Batch 2·3 EDA는 "배치가 얼마나 다른지" 확인용이며, 결과를 보고 feature를 고르면 데이터 누수.
- **Batch 3는 EDA에서만 사용**, Day 2 모델링·평가에는 쓰지 않음 (선택 과제). 보고서에 "Batch 3는 이번에 평가하지 않음"을 명시.
- **과제 설명과 다른 점:** "Batch 1/2 분포가 유사"라고 되어 있지만 실제로는 Batch 2가 훨씬 짧음 (평균 845 vs 566, Batch 2의 72%가 <500). Batch 1로 학습하면 짧은 수명 구간을 거의 못 본다 → 외삽 문제.
- **분류 선택 시 주의:** 기준 550이면 Batch 1은 2%만 단수명, Batch 2는 77%가 단수명 → 클래스 불균형이 반대. 회귀가 더 안전한 근거.
- **Batch 1·2 충전 정책이 거의 안 겹침** (공통 2개). 정책 정보만으로는 일반화가 어려움 → 셀에서 측정되는 신호(ΔQ 등)가 필요.
- `dQ_log_var`와 수명의 상관은 Batch 1 −0.89, Batch 2 −0.90, Batch 3 −0.70. Batch 1·2에서는 유지, Batch 3에서는 약해짐.
- 수명(`cycle_life`)이 NaN인 셀은 Batch 2에 8개, Batch 3에 2개 (VarCharge·SLOWCYCLE 등 특수 실험 셀) → 분석에서 제외.

### 한계
- **knee 탐지는 단순 규칙** (기울기가 초기의 2배가 되는 첫 지점). 정밀하지 않으니 그래프로 직접 확인 후 보고서에 사용.
- **정책 문자열 → 숫자 변환은 근사**: `c_avg = c1×전환SOC + c2×(1−전환SOC)`. 실제 충전 프로파일과 다를 수 있음.
- **ΔQ(V)는 사이클 100 − 10 고정.** 100사이클 미만 셀은 없었지만(최소 101), Batch 2·3는 `Qdlin` 시작점이 달라 단순 비교 시 왜곡 가능 (Q10 곡선 비교 그래프 참고).
- **표본이 작음**: 배치당 약 40~46셀, 정책당 1~3셀. 상관·평균은 경향 수준으로만 해석.
- 위 숫자는 이 노트북을 실행한 결과이므로 전처리·필터 기준을 바꾸면 달라질 수 있음.
